<a href="https://colab.research.google.com/github/leo5358/PL_1142/blob/main/HW4_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 安裝套件

第一次執行時請取消下面 cell 的註解並安裝需要的套件。若已安裝，可直接跳過。

In [17]:
!pip install pandas requests beautifulsoup4 faiss-cpu sentence-transformers gspread google-auth

## 2. 匯入套件與基本設定

In [18]:
import os
import time
import datetime
import requests
import pandas as pd
import numpy as np

from typing import List, Dict, Any, Optional

# Google Sheet 設定
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/1ge-RkD_jfq4NNu5egU2Fk48pBttx6vpIvsPWIh96B_o/edit?usp=sharing"
GOOGLE_WORKSHEET_NAME = "vul_db"

# API URL
NVD_API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"
EPSS_API_URL = "https://api.first.org/data/v1/epss"
CISA_KEV_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"


REQUIRED_COLUMNS = [
    "cve_id",
    "description",
    "published",
    "last_modified",
    "cvss_score",
    "cvss_severity",
    "cwe",
    "vendor",
    "product",
    "epss",
    "kev",
    "known_ransomware",
    "references",
    "risk_priority",
]

## 3. Google Sheets 工具函式

In [19]:
def get_google_worksheet(
    spreadsheet_url: str = GOOGLE_SHEET_URL,
    worksheet_name: str = GOOGLE_WORKSHEET_NAME,
):
    spreadsheet = gc.open_by_url(spreadsheet_url)

    try:
        worksheet = spreadsheet.worksheet(worksheet_name)
    except gspread.WorksheetNotFound:
        worksheet = spreadsheet.add_worksheet(
            title=worksheet_name,
            rows=1000,
            cols=len(REQUIRED_COLUMNS)
        )

    return worksheet


def ensure_google_sheet_schema(
    spreadsheet_url: str = GOOGLE_SHEET_URL,
    worksheet_name: str = GOOGLE_WORKSHEET_NAME,
):
    worksheet = get_google_worksheet(spreadsheet_url, worksheet_name)

    values = worksheet.get_all_values()

    if not values:
        worksheet.append_row(REQUIRED_COLUMNS)
        return "created header"

    header = values[0]

    if header != REQUIRED_COLUMNS:
        worksheet.clear()
        worksheet.append_row(REQUIRED_COLUMNS)
        return "reset header"

    return "ok"


def load_from_google_sheets(
    spreadsheet_url: str = GOOGLE_SHEET_URL,
    worksheet_name: str = GOOGLE_WORKSHEET_NAME,
    fix_schema: bool = True,
) -> pd.DataFrame:
    if fix_schema:
        ensure_google_sheet_schema(spreadsheet_url, worksheet_name)

    worksheet = get_google_worksheet(spreadsheet_url, worksheet_name)
    records = worksheet.get_all_records()

    if not records:
        return pd.DataFrame(columns=REQUIRED_COLUMNS)

    df = pd.DataFrame(records)

    for col in REQUIRED_COLUMNS:
        if col not in df.columns:
            df[col] = ""

    return df[REQUIRED_COLUMNS]


def append_to_google_sheets(
    vuln_df: pd.DataFrame,
    spreadsheet_url: str = GOOGLE_SHEET_URL,
    worksheet_name: str = GOOGLE_WORKSHEET_NAME,
):
    ensure_google_sheet_schema(spreadsheet_url, worksheet_name)
    worksheet = get_google_worksheet(spreadsheet_url, worksheet_name)

    existing_df = load_from_google_sheets(
        spreadsheet_url,
        worksheet_name,
        fix_schema=False
    )

    existing_ids = set(existing_df["cve_id"].astype(str))

    output_df = vuln_df.copy()

    for col in REQUIRED_COLUMNS:
        if col not in output_df.columns:
            output_df[col] = ""

    output_df = output_df[REQUIRED_COLUMNS]
    output_df = output_df[~output_df["cve_id"].astype(str).isin(existing_ids)]

    if output_df.empty:
        print("沒有新的 CVE 需要寫入。")
        return

    output_df = output_df.fillna("").astype(str)

    worksheet.append_rows(
        output_df.values.tolist(),
        value_input_option="USER_ENTERED"
    )

    print(f"新增 {len(output_df)} 筆資料到 Google Sheets。")

## 4. NVD / EPSS / KEV API 函式

In [20]:
def get_nvd_headers() -> Dict[str, str]:
    headers = {}

    api_key = os.getenv("NVD_API_KEY")
    if api_key:
        headers["apiKey"] = api_key

    return headers


def safe_get_json(url: str, params: Optional[dict] = None, headers: Optional[dict] = None, timeout: int = 60):
    try:
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=timeout
        )

        if response.status_code == 429:
            print("遇到 rate limit，等待 30 秒後重試。")
            time.sleep(30)
            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=timeout
            )

        response.raise_for_status()
        return response.json()

    except Exception as e:
        print(f"API request failed: {e}")
        return None


def fetch_nvd_cve(cve_id: str) -> Optional[dict]:
    params = {
        "cveId": cve_id
    }

    data = safe_get_json(
        NVD_API_URL,
        params=params,
        headers=get_nvd_headers()
    )

    if not data:
        return None

    vulns = data.get("vulnerabilities", [])

    if not vulns:
        return None

    return vulns[0]


def fetch_recent_nvd_cves(days: int = 30, max_pages: int = 3) -> List[dict]:
    # 修正原本的 DeprecationWarning，改用 timezone-aware 的 UTC 時間
    end_date = datetime.datetime.now(datetime.UTC)
    start_date = end_date - datetime.timedelta(days=days)

    params = {
        # 將原本的 lastMod 改為 pub，專門抓取「新發布」的漏洞
        "pubStartDate": start_date.strftime("%Y-%m-%dT%H:%M:%S.000"),
        "pubEndDate": end_date.strftime("%Y-%m-%dT%H:%M:%S.000"),
        "resultsPerPage": 1000,
        "startIndex": 0,
    }

    headers = get_nvd_headers()
    results = []

    for page in range(max_pages):
        print(f"正在抓 NVD 第 {page + 1} 頁，startIndex={params['startIndex']}")

        data = safe_get_json(
            NVD_API_URL,
            params=params,
            headers=headers
        )

        if not data:
            break

        vulns = data.get("vulnerabilities", [])
        total = data.get("totalResults", 0)

        print(f"本頁取得 {len(vulns)} 筆，NVD totalResults={total}")

        results.extend(vulns)

        params["startIndex"] += params["resultsPerPage"]

        if params["startIndex"] >= total:
            break

        if os.getenv("NVD_API_KEY"):
            time.sleep(1)
        else:
            time.sleep(6)

    return results

def fetch_epss_batch(cve_ids: List[str]) -> Dict[str, dict]:
    if not cve_ids:
        return {}

    result = {}

    batch_size = 100

    for i in range(0, len(cve_ids), batch_size):
        batch = cve_ids[i:i + batch_size]

        params = {
            "cve": ",".join(batch)
        }

        data = safe_get_json(EPSS_API_URL, params=params)

        if not data:
            continue

        for item in data.get("data", []):
            cve = item.get("cve")
            result[cve] = {
                "epss": item.get("epss", ""),
                "percentile": item.get("percentile", "")
            }

        time.sleep(1)

    return result


def fetch_kev_catalog() -> Dict[str, dict]:
    data = safe_get_json(CISA_KEV_URL)

    if not data:
        return {}

    kev_map = {}

    for item in data.get("vulnerabilities", []):
        cve_id = item.get("cveID")

        if cve_id:
            kev_map[cve_id] = item

    return kev_map

## 5. NVD 資料解析函式

In [21]:
def get_english_description(cve_obj: dict) -> str:
    descriptions = cve_obj.get("descriptions", [])

    for desc in descriptions:
        if desc.get("lang") == "en":
            return desc.get("value", "")

    if descriptions:
        return descriptions[0].get("value", "")

    return ""


def extract_cvss(cve_obj: dict):
    metrics = cve_obj.get("metrics", {})

    metric_keys = [
        "cvssMetricV40",
        "cvssMetricV31",
        "cvssMetricV30",
        "cvssMetricV2",
    ]

    for key in metric_keys:
        if key in metrics and metrics[key]:
            metric = metrics[key][0]
            cvss_data = metric.get("cvssData", {})

            score = cvss_data.get("baseScore", "")
            severity = cvss_data.get("baseSeverity", "")

            if not severity:
                severity = metric.get("baseSeverity", "")

            return score, severity

    return "", ""


def extract_cwe(cve_obj: dict) -> str:
    weaknesses = cve_obj.get("weaknesses", [])

    cwe_list = []

    for weakness in weaknesses:
        for desc in weakness.get("description", []):
            value = desc.get("value", "")
            if value and value not in cwe_list:
                cwe_list.append(value)

    return ", ".join(cwe_list)


def extract_vendor_product(cve_obj: dict):
    configurations = cve_obj.get("configurations", [])

    vendors = set()
    products = set()

    for config in configurations:
        for node in config.get("nodes", []):
            for cpe_match in node.get("cpeMatch", []):
                criteria = cpe_match.get("criteria", "")

                parts = criteria.split(":")

                if len(parts) >= 5:
                    vendor = parts[3]
                    product = parts[4]

                    if vendor and vendor != "*":
                        vendors.add(vendor)

                    if product and product != "*":
                        products.add(product)

    vendor_text = ", ".join(sorted(vendors))
    product_text = ", ".join(sorted(products))

    return vendor_text, product_text


def extract_references(cve_obj: dict) -> str:
    references = cve_obj.get("references", [])

    # NVD API v2: references 通常是 list
    if isinstance(references, list):
        refs = references

    # 舊格式：references 可能是 dict，裡面有 referenceData
    elif isinstance(references, dict):
        refs = references.get("referenceData", [])

    else:
        refs = []

    urls = []

    for ref in refs[:5]:
        if isinstance(ref, dict):
            url = ref.get("url", "")
            if url:
                urls.append(url)

    return "\n".join(urls)


def calculate_risk_priority(cvss_score, epss, kev: bool) -> str:
    try:
        cvss_score = float(cvss_score)
    except:
        cvss_score = 0.0

    try:
        epss = float(epss)
    except:
        epss = 0.0

    if kev:
        return "CRITICAL"

    if cvss_score >= 9.0 and epss >= 0.1:
        return "CRITICAL"

    if cvss_score >= 9.0:
        return "HIGH"

    if epss >= 0.5:
        return "HIGH"

    if cvss_score >= 7.0 or epss >= 0.1:
        return "MEDIUM"

    return "LOW"


def parse_nvd_vulnerability(vuln: dict, epss_map: dict, kev_map: dict) -> dict:
    cve_obj = vuln.get("cve", {})

    cve_id = cve_obj.get("id", "")
    description = get_english_description(cve_obj)

    published = cve_obj.get("published", "")
    last_modified = cve_obj.get("lastModified", "")

    cvss_score, cvss_severity = extract_cvss(cve_obj)
    cwe = extract_cwe(cve_obj)
    vendor, product = extract_vendor_product(cve_obj)
    references = extract_references(cve_obj)

    epss = epss_map.get(cve_id, {}).get("epss", "")

    kev_item = kev_map.get(cve_id)
    kev = kev_item is not None

    known_ransomware = ""
    if kev_item:
        known_ransomware = kev_item.get("knownRansomwareCampaignUse", "")

    risk_priority = calculate_risk_priority(cvss_score, epss, kev)

    return {
        "cve_id": cve_id,
        "description": description,
        "published": published,
        "last_modified": last_modified,
        "cvss_score": cvss_score,
        "cvss_severity": cvss_severity,
        "cwe": cwe,
        "vendor": vendor,
        "product": product,
        "epss": epss,
        "kev": "YES" if kev else "NO",
        "known_ransomware": known_ransomware,
        "references": references,
        "risk_priority": risk_priority,
    }


def normalize_vulnerability_table(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=REQUIRED_COLUMNS)

    output_df = df.copy()

    for col in REQUIRED_COLUMNS:
        if col not in output_df.columns:
            output_df[col] = ""

    output_df = output_df[REQUIRED_COLUMNS]

    output_df["cvss_score"] = pd.to_numeric(output_df["cvss_score"], errors="coerce").fillna(0)
    output_df["epss"] = pd.to_numeric(output_df["epss"], errors="coerce").fillna(0)

    output_df["cve_id"] = output_df["cve_id"].astype(str)

    output_df = output_df.drop_duplicates(subset=["cve_id"], keep="last")

    return output_df

## 6. 爬蟲主流程

In [22]:
def build_dataset_from_nvd_vulns(vulns: List[dict]) -> pd.DataFrame:
    if not vulns:
        return pd.DataFrame(columns=REQUIRED_COLUMNS)

    cve_ids = []

    for vuln in vulns:
        cve_id = vuln.get("cve", {}).get("id", "")
        if cve_id:
            cve_ids.append(cve_id)

    print(f"準備補 EPSS，共 {len(cve_ids)} 筆 CVE。")
    epss_map = fetch_epss_batch(cve_ids)

    print("準備載入 CISA KEV Catalog。")
    kev_map = fetch_kev_catalog()

    rows = []

    for vuln in vulns:
        row = parse_nvd_vulnerability(vuln, epss_map, kev_map)
        rows.append(row)

    df = pd.DataFrame(rows)
    df = normalize_vulnerability_table(df)

    return df


def build_dataset_from_cves(cve_list: List[str]) -> pd.DataFrame:
    vulns = []

    for i, cve_id in enumerate(cve_list):
        print(f"[{i + 1}/{len(cve_list)}] fetching {cve_id}")

        vuln = fetch_nvd_cve(cve_id)

        if vuln:
            vulns.append(vuln)
        else:
            print(f"找不到資料：{cve_id}")

        if os.getenv("NVD_API_KEY"):
            time.sleep(1)
        else:
            time.sleep(6)

    return build_dataset_from_nvd_vulns(vulns)


def fetch_and_sync_to_sheets(cve_list: List[str]) -> pd.DataFrame:
    print(f"正在從 API 抓取 {len(cve_list)} 筆漏洞資料...")

    new_data_df = build_dataset_from_cves(cve_list)

    if new_data_df.empty:
        print("API 沒有抓到任何資料。")
        return pd.DataFrame(columns=REQUIRED_COLUMNS)

    print("正在將資料附加至 Google Sheets...")
    append_to_google_sheets(new_data_df)

    print("同步完成。")
    display(new_data_df)

    return new_data_df


def crawl_recent_vulnerabilities(days: int = 30, max_pages: int = 1, max_sync: Optional[int] = 20) -> pd.DataFrame:
    print(f"開始抓取最近 {days} 天有更新的 CVE。")

    vulns = fetch_recent_nvd_cves(
        days=days,
        max_pages=max_pages
    )

    if not vulns:
        print("沒有從 NVD 抓到任何漏洞。")
        return pd.DataFrame(columns=REQUIRED_COLUMNS)

    if max_sync is not None:
        vulns = vulns[:max_sync]

    print(f"準備整理 {len(vulns)} 筆漏洞資料。")

    df = build_dataset_from_nvd_vulns(vulns)

    if df.empty:
        print("整理後沒有任何資料。")
        return pd.DataFrame(columns=REQUIRED_COLUMNS)

    append_to_google_sheets(df)

    print("最近漏洞同步完成。")
    display(df)

    return df

## 7. RAG 文件轉換

In [23]:
def vulnerability_to_document(row: pd.Series) -> str:
    """
    將漏洞資料轉換為 RAG 文件字串。
    【改動】在文件開頭加入結構化前綴（風險等級、CVE ID、受影響產品），
    讓 embedding 模型更容易抓到最關鍵的資訊，提升向量搜尋的準確性。
    """
    risk_priority = row.get("risk_priority", "")
    cve_id        = row.get("cve_id", "")
    product       = row.get("product", "") or "unknown product"
    kev_flag      = " [KEV]" if str(row.get("kev", "")).upper() == "YES" else ""
    description   = row.get("description", "")

    # 結構化前綴：讓最重要的資訊出現在文件最前面
    prefix = f"[{risk_priority}{kev_flag}] {cve_id} affects {product}: {description}"

    return f"""
{prefix}

CVE ID: {cve_id}

Description:
{description}

Vendor:
{row.get("vendor", "")}

Product:
{product}

CVSS Score:
{row.get("cvss_score", "")}

CVSS Severity:
{row.get("cvss_severity", "")}

EPSS:
{row.get("epss", "")}

CISA KEV:
{row.get("kev", "")}

Known Ransomware:
{row.get("known_ransomware", "")}

CWE:
{row.get("cwe", "")}

Risk Priority:
{risk_priority}

References:
{row.get("references", "")}
""".strip()


## 8. 用 FAISS 建立向量搜尋器

In [24]:
import faiss
from sentence_transformers import SentenceTransformer


class FaissRetriever:
    """
    【改動】將 embedding 模型從 MiniLM-L12 升級為 mpnet-base，
    語意理解能力更強，特別是對資安術語（RCE、SQL Injection、CVE 描述）的處理更準確。
    保留多語言支援（中英文查詢皆可）。
    """
    def __init__(self, model_name: str = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"):
        self.model = SentenceTransformer(model_name)

        self.index = None
        self.documents = []
        self.ids = []
        self.metadatas = []
        self.dimension = None

    def build(self, documents: list[str], ids: list[str], metadatas: list[dict]):
        self.documents = documents
        self.ids = ids
        self.metadatas = metadatas

        if not documents:
            self.index = None
            self.dimension = None
            return

        embeddings = self.model.encode(
            documents,
            convert_to_numpy=True,
            show_progress_bar=True
        ).astype("float32")

        # normalize 後用 Inner Product，等價於 cosine similarity
        faiss.normalize_L2(embeddings)

        self.dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(self.dimension)
        self.index.add(embeddings)

        print(f"FAISS index 建立完成，共 {self.index.ntotal} 筆資料，dimension={self.dimension}")

    def search(self, query: str, top_k: int = 5):
        if self.index is None or self.index.ntotal == 0:
            return []

        query_embedding = self.model.encode(
            [query],
            convert_to_numpy=True
        ).astype("float32")

        faiss.normalize_L2(query_embedding)

        scores, indices = self.index.search(query_embedding, top_k)

        results = []

        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue

            results.append({
                "score": float(score),
                "id": self.ids[idx],
                "metadata": self.metadatas[idx],
                "document": self.documents[idx],
            })

        return results


## 9. 建立與更新 RAG Index

In [25]:
df = pd.DataFrame(columns=REQUIRED_COLUMNS)
documents = []
ids = []
metadatas = []
retriever = FaissRetriever()


def refresh_rag_index():
    global df, documents, ids, metadatas, retriever

    df = load_from_google_sheets()
    df = normalize_vulnerability_table(df)

    if df.empty:
        print("Google Sheets 裡目前沒有資料，無法建立 RAG index。")
        return

    documents = [vulnerability_to_document(row) for _, row in df.iterrows()]
    ids = df["cve_id"].tolist()

    metadatas = df[
        [
            "cve_id",
            "product",
            "cvss_severity",
            "kev",
            "cwe",
            "risk_priority",
        ]
    ].to_dict("records")

    retriever.build(documents, ids, metadatas)

    print(f"RAG 系統已更新，目前共有 {len(df)} 筆資料。")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
def search_vulnerabilities(query: str, top_k: int = 5):
    results = retriever.search(query, top_k=top_k)

    if not results:
        return "目前沒有可搜尋的資料，請先執行 crawl_recent_vulnerabilities() 或 fetch_and_sync_to_sheets()。"

    output = []

    for i, item in enumerate(results, start=1):
        meta = item["metadata"]

        text = f"""
## Result {i}

Score: {item["score"]:.4f}

CVE: {meta.get("cve_id", "")}
Product: {meta.get("product", "")}
Severity: {meta.get("cvss_severity", "")}
KEV: {meta.get("kev", "")}
CWE: {meta.get("cwe", "")}
Risk Priority: {meta.get("risk_priority", "")}

{item["document"]}
""".strip()

        output.append(text)

    return "\n\n---\n\n".join(output)


## 10. 第一次測試：確認 Google Sheet 可用

In [27]:
# 1. 進行 Google Colab 帳號授權
from google.colab import auth
auth.authenticate_user()

# 2. 匯入 gspread 並建立 gc (Google Client) 物件
import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

schema_result = ensure_google_sheet_schema()
print("Schema:", schema_result)

loaded_df = load_from_google_sheets()
print("目前 Google Sheets 資料筆數:", len(loaded_df))
display(loaded_df.head())

Schema: ok
目前 Google Sheets 資料筆數: 205


,cve_id,description,published,last_modified,cvss_score,cvss_severity,cwe,vendor,product,epss,kev,known_ransomware,references,risk_priority
0,CVE-2024-42008,A Cross-Site Scripting vulnerability in rcmail...,2024-08-05 19:15:38,2025-03-13 16:15:21,9.3,CRITICAL,CWE-79,roundcube,webmail,0.50951,NO,,https://github.com/roundcube/roundcubemail/rel...,CRITICAL
1,CVE-2023-38831,RARLAB WinRAR before 6.23 allows attackers to ...,2023-08-23 17:15:44,2025-10-31 14:39:34,7.8,HIGH,"CWE-345, CWE-351",rarlab,winrar,0.93664,YES,Known,http://packetstormsecurity.com/files/174573/Wi...,CRITICAL
2,CVE-2024-21413,Microsoft Outlook Remote Code Execution Vulner...,2024-02-13 18:16:00,2025-10-28 14:36:11,9.8,CRITICAL,"CWE-20, NVD-CWE-noinfo",microsoft,"365_apps, office_2016, office_2019, office_lon...",0.92992,YES,Unknown,https://msrc.microsoft.com/update-guide/vulner...,CRITICAL
3,CVE-2021-41773,A flaw was found in a change made to path norm...,2021-10-05 9:15:08,2026-02-17 19:49:26,9.8,CRITICAL,CWE-22,"apache, fedoraproject, netapp, oracle","cloud_backup, fedora, http_server, instantis_e...",0.94391,YES,Known,http://packetstormsecurity.com/files/164418/Ap...,CRITICAL
4,CVE-2023-34362,In Progress MOVEit Transfer before 2021.0.6 (1...,2023-06-02 14:15:09,2025-10-27 14:37:08,9.8,CRITICAL,CWE-89,progress,"moveit_cloud, moveit_transfer",0.94254,YES,Known,http://packetstormsecurity.com/files/172883/MO...,CRITICAL


## 11. 手動抓幾筆確定存在的 CVE

In [28]:
test_df = fetch_and_sync_to_sheets([
    "CVE-2024-42008",
    "CVE-2023-38831",
    "CVE-2024-21413",
    "CVE-2021-41773",
    "CVE-2023-34362",
])

正在從 API 抓取 5 筆漏洞資料...
[1/5] fetching CVE-2024-42008
[2/5] fetching CVE-2023-38831
[3/5] fetching CVE-2024-21413
[4/5] fetching CVE-2021-41773
[5/5] fetching CVE-2023-34362
準備補 EPSS，共 5 筆 CVE。
準備載入 CISA KEV Catalog。
正在將資料附加至 Google Sheets...
沒有新的 CVE 需要寫入。
同步完成。


,cve_id,description,published,last_modified,cvss_score,cvss_severity,cwe,vendor,product,epss,kev,known_ransomware,references,risk_priority
0,CVE-2024-42008,A Cross-Site Scripting vulnerability in rcmail...,2024-08-05T19:15:38.153,2025-03-13T16:15:21.240,9.3,CRITICAL,CWE-79,roundcube,webmail,0.50951,NO,,https://github.com/roundcube/roundcubemail/rel...,CRITICAL
1,CVE-2023-38831,RARLAB WinRAR before 6.23 allows attackers to ...,2023-08-23T17:15:43.863,2025-10-31T14:39:33.660,7.8,HIGH,"CWE-345, CWE-351",rarlab,winrar,0.93664,YES,Known,http://packetstormsecurity.com/files/174573/Wi...,CRITICAL
2,CVE-2024-21413,Microsoft Outlook Remote Code Execution Vulner...,2024-02-13T18:16:00.137,2025-10-28T14:36:10.643,9.8,CRITICAL,"CWE-20, NVD-CWE-noinfo",microsoft,"365_apps, office_2016, office_2019, office_lon...",0.92992,YES,Unknown,https://msrc.microsoft.com/update-guide/vulner...,CRITICAL
3,CVE-2021-41773,A flaw was found in a change made to path norm...,2021-10-05T09:15:07.593,2026-02-17T19:49:26.367,9.8,CRITICAL,CWE-22,"apache, fedoraproject, netapp, oracle","cloud_backup, fedora, http_server, instantis_e...",0.94391,YES,Known,http://packetstormsecurity.com/files/164418/Ap...,CRITICAL
4,CVE-2023-34362,In Progress MOVEit Transfer before 2021.0.6 (1...,2023-06-02T14:15:09.487,2025-10-27T14:37:08.460,9.8,CRITICAL,CWE-89,progress,"moveit_cloud, moveit_transfer",0.94254,YES,Known,http://packetstormsecurity.com/files/172883/MO...,CRITICAL


## 12. 建立 RAG index

In [29]:
refresh_rag_index()

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

FAISS index 建立完成，共 205 筆資料，dimension=768
RAG 系統已更新，目前共有 205 筆資料。


## 13. 測試搜尋

In [30]:
print(search_vulnerabilities("WinRAR 有哪些高風險漏洞？", top_k=3))

## Result 1

Score: 0.6495

CVE: CVE-2023-38831
Product: winrar
Severity: HIGH
KEV: YES
CWE: CWE-345, CWE-351
Risk Priority: CRITICAL

[CRITICAL [KEV]] CVE-2023-38831 affects winrar: RARLAB WinRAR before 6.23 allows attackers to execute arbitrary code when a user attempts to view a benign file within a ZIP archive. The issue occurs because a ZIP archive may include a benign file (such as an ordinary .JPG file) and also a folder that has the same name as the benign file, and the contents of the folder (which may include executable content) are processed during an attempt to access only the benign file. This was exploited in the wild in April through October 2023.

CVE ID: CVE-2023-38831

Description:
RARLAB WinRAR before 6.23 allows attackers to execute arbitrary code when a user attempts to view a benign file within a ZIP archive. The issue occurs because a ZIP archive may include a benign file (such as an ordinary .JPG file) and also a folder that has the same name as the benign file,

## 14. 真的開始爬最近漏洞


In [31]:
recent_df = crawl_recent_vulnerabilities(
    days=30,
    max_pages=2,
    max_sync=100
)

開始抓取最近 30 天有更新的 CVE。
正在抓 NVD 第 1 頁，startIndex=0
本頁取得 1000 筆，NVD totalResults=6900
正在抓 NVD 第 2 頁，startIndex=1000
本頁取得 1000 筆，NVD totalResults=6900
準備整理 100 筆漏洞資料。
準備補 EPSS，共 100 筆 CVE。
準備載入 CISA KEV Catalog。
沒有新的 CVE 需要寫入。
最近漏洞同步完成。


,cve_id,description,published,last_modified,cvss_score,cvss_severity,cwe,vendor,product,epss,kev,known_ransomware,references,risk_priority
0,CVE-2026-31720,"In the Linux kernel, the following vulnerabili...",2026-05-01T15:16:34.360,2026-05-06T20:58:09.417,7.8,HIGH,CWE-787,linux,linux_kernel,0.00013,NO,,https://git.kernel.org/stable/c/0d41772d98dcaf...,MEDIUM
1,CVE-2026-31721,"In the Linux kernel, the following vulnerabili...",2026-05-01T15:16:34.490,2026-05-06T20:56:34.973,5.5,MEDIUM,NVD-CWE-noinfo,linux,linux_kernel,0.00013,NO,,https://git.kernel.org/stable/c/13440c0db227c5...,LOW
2,CVE-2026-31722,"In the Linux kernel, the following vulnerabili...",2026-05-01T15:16:34.617,2026-05-06T20:55:31.143,5.5,MEDIUM,NVD-CWE-noinfo,linux,linux_kernel,0.00013,NO,,https://git.kernel.org/stable/c/18ada801899f2b...,LOW
3,CVE-2026-31723,"In the Linux kernel, the following vulnerabili...",2026-05-01T15:16:34.727,2026-05-07T17:03:30.180,5.5,MEDIUM,NVD-CWE-noinfo,linux,linux_kernel,0.00013,NO,,https://git.kernel.org/stable/c/06524cd1c9011b...,LOW
4,CVE-2026-31724,"In the Linux kernel, the following vulnerabili...",2026-05-01T15:16:34.833,2026-05-07T17:00:23.647,5.5,MEDIUM,NVD-CWE-noinfo,linux,linux_kernel,0.00013,NO,,https://git.kernel.org/stable/c/14730506b9e2a0...,LOW
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,CVE-2026-43029,"In the Linux kernel, the following vulnerabili...",2026-05-01T15:16:47.427,2026-05-08T18:33:39.740,7.5,HIGH,CWE-667,linux,linux_kernel,0.00045,NO,,https://git.kernel.org/stable/c/58b58b9ba89c43...,MEDIUM
96,CVE-2026-43030,"In the Linux kernel, the following vulnerabili...",2026-05-01T15:16:47.557,2026-05-08T18:36:14.140,7.8,HIGH,NVD-CWE-noinfo,linux,linux_kernel,0.00013,NO,,https://git.kernel.org/stable/c/015a74476dc1ab...,MEDIUM
97,CVE-2026-43031,"In the Linux kernel, the following vulnerabili...",2026-05-01T15:16:47.680,2026-05-08T18:38:07.040,7.5,HIGH,NVD-CWE-noinfo,linux,linux_kernel,0.00050,NO,,https://git.kernel.org/stable/c/2a0323a913109b...,MEDIUM
98,CVE-2026-43032,"In the Linux kernel, the following vulnerabili...",2026-05-01T15:16:47.787,2026-05-08T18:39:32.083,5.5,MEDIUM,NVD-CWE-noinfo,linux,linux_kernel,0.00013,NO,,https://git.kernel.org/stable/c/23e925183db26c...,LOW


In [32]:
refresh_rag_index()

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

FAISS index 建立完成，共 205 筆資料，dimension=768
RAG 系統已更新，目前共有 205 筆資料。


In [33]:
print(search_vulnerabilities("最近有哪些最值得優先修補的漏洞？", top_k=5))

## Result 1

Score: 0.4108

CVE: CVE-2026-43028
Product: linux_kernel
Severity: HIGH
KEV: NO
CWE: NVD-CWE-noinfo
Risk Priority: MEDIUM

[MEDIUM] CVE-2026-43028 affects linux_kernel: In the Linux kernel, the following vulnerability has been resolved:

netfilter: x_tables: ensure names are nul-terminated

Reject names that lack a \0 character before feeding them
to functions that expect c-strings.

Fixes tag is the most recent commit that needs this change.

CVE ID: CVE-2026-43028

Description:
In the Linux kernel, the following vulnerability has been resolved:

netfilter: x_tables: ensure names are nul-terminated

Reject names that lack a \0 character before feeding them
to functions that expect c-strings.

Fixes tag is the most recent commit that needs this change.

Vendor:
linux

Product:
linux_kernel

CVSS Score:
7.1

CVSS Severity:
HIGH

EPSS:
0.00013

CISA KEV:
NO

Known Ransomware:


CWE:
NVD-CWE-noinfo

Risk Priority:
MEDIUM

References:
https://git.kernel.org/stable/c/673bbd36cb

In [38]:
# severity 嚴重程度排序對應表（用於下限過濾）
SEVERITY_ORDER = ["LOW", "NONE", "MEDIUM", "HIGH", "CRITICAL"]


def parse_query_filters(query: str) -> dict:
    """
    解析使用者的自然語言查詢，提取初步過濾條件。
    【改動】改為「下限」過濾邏輯，避免過度限縮搜尋範圍：
      - 查詢含「嚴重/CRITICAL」→ 保留 CRITICAL
      - 查詢含「高風險/HIGH」   → 保留 HIGH + CRITICAL
      - 查詢含「中等/MEDIUM」   → 保留 MEDIUM + HIGH + CRITICAL
    原本精確比對會漏掉鄰近等級的結果，改為下限後覆蓋更完整。
    """
    filters = {}
    query_lower = query.lower()

    # 判斷 CVSS 嚴重程度（下限）
    if "critical" in query_lower or "嚴重" in query_lower:
        filters["cvss_severity_min"] = "CRITICAL"
    elif "high" in query_lower or "高風險" in query_lower:
        filters["cvss_severity_min"] = "HIGH"
    elif "medium" in query_lower or "中等" in query_lower:
        filters["cvss_severity_min"] = "MEDIUM"

    # 判斷 CISA KEV 狀態
    if "kev" in query_lower or "利用" in query_lower or "exploited" in query_lower:
        filters["kev"] = "YES"

    return filters


def apply_filters(df: pd.DataFrame, filters: dict) -> pd.DataFrame:
    """
    根據解析出的條件過濾 DataFrame。
    【改動】cvss_severity 改用下限過濾（≥ 指定等級），而非精確比對，
    避免「查 HIGH 但漏掉 CRITICAL」的問題。
    """
    filtered_df = df.copy()

    # 處理 severity 下限過濾
    if "cvss_severity_min" in filters:
        min_severity = filters["cvss_severity_min"].upper()
        if min_severity in SEVERITY_ORDER:
            min_idx = SEVERITY_ORDER.index(min_severity)
            # 保留等級 >= min_severity 的資料
            filtered_df = filtered_df[
                filtered_df["cvss_severity"].astype(str).str.upper().apply(
                    lambda s: SEVERITY_ORDER.index(s) >= min_idx
                    if s in SEVERITY_ORDER else False
                )
            ]

    # 處理 KEV 精確過濾
    if "kev" in filters:
        filtered_df = filtered_df[
            filtered_df["kev"].astype(str).str.upper() == filters["kev"].upper()
        ]

    return filtered_df


SIMILARITY_THRESHOLD = 0.25  # 可依資料集調整

def rerank_results(results: list, df: pd.DataFrame) -> list:
    cve_lookup = df.set_index("cve_id") if not df.empty else pd.DataFrame()
    severity_bonus = {"CRITICAL": 0.20, "HIGH": 0.10, "MEDIUM": 0.05}

    scored = []
    for r in results:
        # 新增：語意相似度太低直接過濾，避免無關結果被風險加權推上來
        if r["score"] < SIMILARITY_THRESHOLD:
            continue

        cve_id = r["metadata"].get("cve_id", "")
        epss = 0.0
        if cve_id and cve_id in cve_lookup.index:
            try:
                epss = float(cve_lookup.at[cve_id, "epss"] or 0)
            except (ValueError, KeyError):
                epss = 0.0

        kev_bonus      = 0.30 if r["metadata"].get("kev", "").upper() == "YES" else 0.0
        severity       = r["metadata"].get("cvss_severity", "").upper()
        severity_score = severity_bonus.get(severity, 0.0)

        r["rerank_score"] = r["score"] + epss * 0.50 + kev_bonus + severity_score
        scored.append(r)

    return sorted(scored, key=lambda x: x["rerank_score"], reverse=True)


def format_search_results(results: list) -> str:
    """
    將搜尋結果格式化為易於閱讀的 Markdown 字串。
    【改動】輸出同時顯示 FAISS score 與 rerank_score，方便 debug。
    """
    if not results:
        return "找不到符合條件的結果。"

    output = []
    for i, item in enumerate(results, start=1):
        meta = item["metadata"]
        rerank_score = item.get("rerank_score", item["score"])
        text = f"""
## Result {i}

Rerank Score : {rerank_score:.4f}  (FAISS: {item["score"]:.4f})

CVE          : {meta.get("cve_id", "")}
Product      : {meta.get("product", "")}
Severity     : {meta.get("cvss_severity", "")}
KEV          : {meta.get("kev", "")}
CWE          : {meta.get("cwe", "")}
Risk Priority: {meta.get("risk_priority", "")}

{item["document"]}
""".strip()
        output.append(text)

    return "\n\n---\n\n".join(output)


In [35]:
# 舊版 smart_search 已移除（原本被 triple-quote 包住的草稿版本）
# 請使用下方 Cell 33 的正式版 smart_search。


In [39]:
def search_vulnerabilities(query: str, top_k: int = 5) -> str:
    """單純向量搜尋（不做 pre-filter），適合一般關鍵字查詢。"""
    results = retriever.search(query, top_k=top_k)

    if not results:
        return "找不到合適的結果"

    # 加上 reranking（利用全局 df）
    results = rerank_results(results, df)
    return format_search_results(results)


def smart_search(query: str, top_k: int = 5) -> str:
    """
    智慧搜尋：先依關鍵字做 metadata 過濾，再對候選子集做向量搜尋，最後重排序。
    【改動】不再每次重建 FaissRetriever，改用 FAISS ID selector 在全局 index
    上直接限制搜尋範圍，大幅降低重複 encode 的開銷。
    """
    filters = parse_query_filters(query)
    candidate_df = apply_filters(df, filters) if filters else df

    if candidate_df.empty:
        return "找不到合適的結果"

    # 取得候選 CVE 的 index 位置（對應 retriever.ids）
    candidate_ids = set(candidate_df["cve_id"].tolist())
    allowed_indices = [
        i for i, cve in enumerate(retriever.ids) if cve in candidate_ids
    ]

    if not allowed_indices:
        return "找不到合適的結果"

    # 【核心改動】用 IDSelectorArray 限制搜尋範圍，在已有 index 上做子集搜尋
    # 不重建 retriever，不重新 encode 文件，效能大幅提升
    query_embedding = retriever.model.encode(
        [query], convert_to_numpy=True
    ).astype("float32")
    faiss.normalize_L2(query_embedding)

    selector = faiss.IDSelectorArray(np.array(allowed_indices, dtype=np.int64))
    search_params = faiss.SearchParametersIVF()
    search_params.sel = selector

    # IndexFlatIP 使用 SearchParameters 進行子集搜尋
    actual_k = min(top_k * 5, len(allowed_indices))
    scores, indices = retriever.index.search(
        query_embedding,
        actual_k,
        params=search_params
    )

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append({
            "score": float(score),
            "id": retriever.ids[idx],
            "metadata": retriever.metadatas[idx],
            "document": retriever.documents[idx],
        })

    if not results:
        return "找不到合適的結果"

    results = rerank_results(results, candidate_df)
    return format_search_results(results[:top_k])


In [41]:
search_str = str(input("搜尋："))
#print(search_vulnerabilities(search_str, top_k=5))

print("===========================\n")
print(smart_search(search_str, top_k=5))

搜尋：linux kernel

## Result 1

Rerank Score : 0.7090  (FAISS: 0.6090)

CVE          : CVE-2026-31745
Product      : linux_kernel
Severity     : HIGH
KEV          : NO
CWE          : CWE-415
Risk Priority: MEDIUM

[MEDIUM] CVE-2026-31745 affects linux_kernel: In the Linux kernel, the following vulnerability has been resolved:

reset: gpio: fix double free in reset_add_gpio_aux_device() error path

When __auxiliary_device_add() fails, reset_add_gpio_aux_device()
calls auxiliary_device_uninit(adev).

The device release callback reset_gpio_aux_device_release() frees
adev, but the current error path then calls kfree(adev) again,
causing a double free.

Keep kfree(adev) for the auxiliary_device_init() failure path, but
avoid freeing adev after auxiliary_device_uninit().

CVE ID: CVE-2026-31745

Description:
In the Linux kernel, the following vulnerability has been resolved:

reset: gpio: fix double free in reset_add_gpio_aux_device() error path

When __auxiliary_device_add() fails, reset_add_